<a href="https://colab.research.google.com/github/Realosunboy6/550-Stocks-Portfolio-Theory-Python-Julia-/blob/main/Portfolio_Optimization_Colab(2026%20%231).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Portfolio Optimization — Full Production Pipeline
### GICS 11-Sector Universe + Sector ETFs · 2021-2026

| Phase | Module | Deliverable |
|-------|--------|-------------|
| 1 | Data preparation | Clean returns matrix |
| 2 | Markowitz baseline | GMV · Max-Sharpe · Efficient frontier |
| 3 | Robust covariance | Sample vs EWM vs Ledoit-Wolf |
| 4 | Constraints & costs | Weight bounds · turnover · TC |
| 5 | Rolling backtest | 6-strategy out-of-sample comparison |
| 6 | CVaR optimizer | Tail-risk constrained portfolios |
| 7 | Stress testing | Historical + hypothetical scenarios |
| 8 | Final result | Winner portfolio · dashboard · sector allocation |

> All charts are interactive Plotly. Run cells top-to-bottom.


In [28]:
!pip install -q yfinance>=0.2.40 cvxpy clarabel scs plotly scikit-learn
print("Install complete.")

Install complete.


In [29]:
import warnings, time
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import cvxpy  as cp
import plotly.graph_objects as go
import plotly.express       as px
from plotly.subplots        import make_subplots
from sklearn.covariance     import LedoitWolf
import yfinance as yf

np.random.seed(42)

CFG = dict(
    start         = "2021-01-01",
    end           = "2026-04-11",
    rf            = 0.04,     # annualised risk-free rate
    min_history   = 0.75,     # keep assets with >= 75 % non-NaN rows
    weight_cap    = 0.05,     # max weight per asset
    tc_bps        = 10,       # one-way transaction cost (bps)
    train_window  = 252,      # rolling training window (days)
    rebal_freq    = 21,       # rebalance every ~1 month
    frontier_pts  = 15,       # efficient-frontier grid points
    batch_size    = 50,       # yfinance batch size
)

print("Config ready")
print(f"  Universe : GICS 11 sectors + 11 ETFs  |  {CFG['start']} to {CFG['end']}")
print(f"  rf={CFG['rf']*100:.1f}%  weight_cap={CFG['weight_cap']*100:.0f}%  tc={CFG['tc_bps']}bps")


Config ready
  Universe : GICS 11 sectors + 11 ETFs  |  2021-01-01 to 2026-04-11
  rf=4.0%  weight_cap=5%  tc=10bps


## Phase 1 · Data Preparation

We compute two types of returns from the price series for each asset.

**Log returns**  
`r_t = ln(P_t / P_{t-1})`  

Use log returns for estimation because they are additive over time and work well for statistical modeling.

**Simple returns**  
`r_t = (P_t - P_{t-1}) / P_{t-1}`  

Use simple returns in the backtest because they represent the actual percentage gain or loss and match portfolio profit and loss directly.

In [30]:
# ── Universe definition ───────────────────────────────────────
SECTORS = {
    "Communication Services": [
        "LYV","GOOG","META","DIS","NFLX","T","VZ","TMUS","CABO","NWSA",
        "TTWO","EA","NXST","FOX","WBD","ROKU","CHTR","OMC","ZG","SIRI",
        "SPOT","TCEHY","CCOI","PINS","SNAP","BIDU","NTES","SE","BILI","CMCSA",
        "ZM","VOD","RCI","LUMN","AMX","SKM","NTTYY",
    ],
    "Consumer Discretionary": [
        "AMZN","TSLA","HD","MCD","NKE","LOW","SBUX","TJX","TGT","BKNG",
        "GM","F","ROST","MAR","LVS","ORLY","YUM","EBAY","AZO","DHI",
        "LEN","RCL","HLT","CMG","NVR","DRI","ULTA","DG","KMX","BBY",
        "ETSY","HAS","GRMN","MGM","WHR","TPR","RL","GPC","WSM","BURL",
        "POOL","CZR","WYNN",
    ],
    "Consumer Staples": [
        "PG","KO","PEP","WMT","PM","MO","COST","MDLZ","UL","CL",
        "KHC","KMB","GIS","STZ","ADM","MNST","SYY","EL","TAP","CPB",
        "CAG","HSY","TSN","MKC","CHD","KDP","CLX","LW","HRL","SJM",
        "BG","TSCO","SFM","KR","SAM","POST","INGR","FLO",
    ],
    "Energy": [
        "XOM","CVX","BP","EQT","COP","EOG","PBR","KMI","PSX","VLO",
        "MPC","WMB","SU","OXY","TRGP","CTRA","APA","DVN","SLB","LNG",
        "OVV","AR","RRC","HAL","BKR","SM","MTDR","PAA","MUR","NOG","OKE",
    ],
    "Financials": [
        "JPM","BAC","WFC","C","AXP","BK","BLK","SCHW","USB","PNC",
        "TFC","COF","STT","CB","AIG","MET","PRU","ALL","TRV","AFL",
        "PFG","AMP","CME","ICE","SPGI","MCO","NDAQ","V","MA","PYPL",
        "AON","MS","GS","SYF","ALLY","KEY","RF","FITB","HBAN",
    ],
    "Health Care": [
        "JNJ","UNH","PFE","ABBV","LLY","MRK","TMO","ABT","DHR","BMY",
        "AMGN","MDT","GILD","CVS","ISRG","CNC","ELV","SYK","ZTS","REGN",
        "VRTX","BSX","BDX","ILMN","BIIB","EW","IQV","MCK","CI","NVO",
        "CAH","HOLX","RMD","HCA","LH","DGX","IDXX","DVA","UHS","HUM",
    ],
    "Industrials": [
        "HON","UNP","UPS","RTX","CAT","BA","DE","GE","MMM","LMT",
        "FDX","NOC","GD","EMR","ITW","ETN","CSX","WM","CMI","WAB",
        "LHX","GWW","PH","ROP","TT","DAL","AAL","UAL","SWK","FAST",
        "RSG","CTAS","XYL","IR","PNR","ROL","SNA","URI","JBHT","DOV","PWR",
    ],
    "Information Technology": [
        "AAPL","MSFT","NVDA","AVGO","ORCL","CSCO","ACN","ADBE","TXN","IBM",
        "CRM","AMD","INTC","QCOM","AMAT","INTU","ADI","MU","LRCX","NOW",
        "PAYX","ADP","CTSH","CDNS","SNPS","KLAC","MCHP","NXPI","ADSK","FTNT",
        "PANW","HPQ","KEYS","GLW","STX","WDAY","SWKS","MPWR","ON","APH",
    ],
    "Materials": [
        "LIN","SHW","APD","ECL","NEM","FCX","DD","VMC","MLM","LYB",
        "PPG","DOW","NUE","IFF","ALB","EMN","MOS","FMC","AVY","CF",
        "AA","IP","PKG","BALL","HUN","RS","STLD","CLF","RPM","OLN",
        "OC","SQM",
    ],
    "Real Estate": [
        "PLD","AMT","EQIX","SPG","PSA","CCI","WELL","O","DLR","EXR",
        "AVB","EQR","ESS","VTR","CUBE","MAA","INVH","UDR","SUI","CPT",
        "WY","KIM","FRT","REG","IRM","HST","VNO","NNN","BRX","STAG",
        "OHI","LTC","BXP","REXR","ADC","ARE",
    ],
    "Utilities": [
        "NEE","SO","DUK","D","AEP","EXC","SRE","XEL","PEG","ED",
        "WEC","ES","EIX","DTE","PPL","FE","AEE","CMS","ATO","NI",
        "CNP","EVRG","LNT","PNW","AES","NRG","VST","BKH","IDA","UGI",
        "AWK","HE",
    ],
}

SECTOR_ETFS = {
    "Communication Services" : "XLC",
    "Consumer Discretionary" : "XLY",
    "Consumer Staples"        : "XLP",
    "Energy"                  : "XLE",
    "Financials"              : "XLF",
    "Health Care"             : "XLV",
    "Industrials"             : "XLI",
    "Information Technology"  : "XLK",
    "Materials"               : "XLB",
    "Real Estate"             : "XLRE",
    "Utilities"               : "XLU",
}

# Build flat list + sector map
ALL_TICKERS = []
SECTOR_MAP  = {}
for sec, tickers in SECTORS.items():
    for t in tickers:
        if t not in SECTOR_MAP:
            ALL_TICKERS.append(t)
            SECTOR_MAP[t] = sec
    etf = SECTOR_ETFS[sec]
    if etf not in SECTOR_MAP:
        ALL_TICKERS.append(etf)
        SECTOR_MAP[etf] = sec + " ETF"

ALL_TICKERS = list(dict.fromkeys(ALL_TICKERS))
print(f"Unique tickers in universe : {len(ALL_TICKERS)}")
print(f"  ({len(SECTORS)} sectors  +  {len(SECTOR_ETFS)} sector ETFs)")


Unique tickers in universe : 420
  (11 sectors  +  11 sector ETFs)


In [31]:
# ── Download ──────────────────────────────────────────────────
def download_prices(tickers, start, end, batch=50):
    frames, n = [], len(tickers)
    for i in range(0, n, batch):
        chunk = tickers[i:i+batch]
        label = f"[{i+1}-{min(i+batch,n)}/{n}]"
        try:
            raw = yf.download(chunk, start=start, end=end,
                              progress=False, auto_adjust=True)
            if raw.empty:
                print(f"  {label}  empty"); continue
            if isinstance(raw.columns, pd.MultiIndex):
                px = raw["Close"]
            else:
                px = raw[["Close"]] if "Close" in raw.columns else raw
            if isinstance(px, pd.Series):
                px = px.to_frame(chunk[0])
            frames.append(px)
            print(f"  {label}  OK  {px.shape[1]} cols")
        except Exception as e:
            print(f"  {label}  FAIL  {e}")
        if i + batch < n:
            time.sleep(0.4)
    if not frames:
        return None
    out = pd.concat(frames, axis=1)
    out = out.loc[:, ~out.columns.duplicated()]
    # strip columns that are entirely NaN (delisted tickers)
    before = out.shape[1]
    out = out.dropna(axis=1, how="all")
    if before - out.shape[1]:
        print(f"  Removed {before-out.shape[1]} all-NaN tickers after concat.")
    return out


print(f"Downloading {len(ALL_TICKERS)} tickers  ({CFG['start']} to {CFG['end']}) ...")
RAW_PRICES = download_prices(ALL_TICKERS, CFG["start"], CFG["end"],
                              batch=CFG["batch_size"])
if RAW_PRICES is None or RAW_PRICES.empty:
    raise RuntimeError("Download returned nothing. Check yfinance version / network.")
print(f"Raw shape : {RAW_PRICES.shape}")


  [1-50/420]  OK  50 cols
  [51-100/420]  OK  50 cols
  [101-150/420]  OK  50 cols
  [151-200/420]  OK  50 cols
  [201-250/420]  OK  50 cols
  [251-300/420]  OK  50 cols
  [301-350/420]  OK  50 cols
  [351-400/420]  OK  50 cols
  [401-420/420]  OK  20 cols
Raw shape : (1323, 420)


In [32]:
# ── Returns & cleaning ────────────────────────────────────────
def log_ret(px):
    # how='all' so a single all-NaN column never wipes every row
    return np.log(px / px.shift(1)).dropna(how="all")

def simple_ret(px):
    return px.pct_change().dropna(how="all")

def clean(rets, min_cov=0.75, vol_floor=1e-6):
    cov_frac = rets.notna().mean()
    keep = cov_frac[cov_frac >= min_cov].index
    n_drop = rets.shape[1] - len(keep)
    if n_drop:
        print(f"  Coverage filter: dropped {n_drop}  ({len(keep)} remain)")
    out = rets[keep].copy()
    out = out.loc[:, out.std() > vol_floor]
    out = out.fillna(0.0)
    if out.shape[1] == 0:
        raise RuntimeError(
            "All assets dropped by coverage filter. "
            "Lower min_history in CFG or check download."
        )
    return out


LOG_RETS    = log_ret(RAW_PRICES)
SIMPLE_RETS = simple_ret(RAW_PRICES)
RETS        = clean(LOG_RETS, min_cov=CFG["min_history"])

ASSETS      = RETS.columns.tolist()
STAGS       = [SECTOR_MAP.get(t, "Unknown") for t in ASSETS]
N, T        = len(ASSETS), len(RETS)
TN          = T / N

flag = "Ledoit-Wolf mandatory" if TN < 5 else "OK"
print("=" * 52)
print(f"  Assets        : {N}")
print(f"  Trading days  : {T}")
print(f"  Date range    : {RETS.index[0].date()} - {RETS.index[-1].date()}")
print(f"  T/N ratio     : {TN:.1f}  [{flag}]")
print("=" * 52)

sec_counts = pd.Series(STAGS).value_counts().sort_index()
print("\nAssets per sector:")
for s, c in sec_counts.items():
    print(f"  {s:<35} {c:>3}")


  Assets        : 420
  Trading days  : 1322
  Date range    : 2021-01-05 - 2026-04-10
  T/N ratio     : 3.1  [Ledoit-Wolf mandatory]

Assets per sector:
  Communication Services               37
  Communication Services ETF            1
  Consumer Discretionary               43
  Consumer Discretionary ETF            1
  Consumer Staples                     38
  Consumer Staples ETF                  1
  Energy                               31
  Energy ETF                            1
  Financials                           39
  Financials ETF                        1
  Health Care                          40
  Health Care ETF                       1
  Industrials                          41
  Industrials ETF                       1
  Information Technology               40
  Information Technology ETF            1
  Materials                            32
  Materials ETF                         1
  Real Estate                          36
  Real Estate ETF                       1
  Util

In [33]:
# ── Data visualisations ──────────────────────────────────────

# 1. Sector composition pie
base_secs = [s.replace(" ETF","") for s in STAGS]
sc = pd.Series(base_secs).value_counts()
fig = go.Figure(go.Pie(
    labels=sc.index, values=sc.values, hole=0.38,
    textinfo="label+percent",
    marker=dict(colors=px.colors.qualitative.Set3)))
fig.update_layout(
    title="<b>Universe — GICS Sector Composition</b>",
    template="plotly_white", height=460)
fig.show()

# 2. Normalised price percentile band
norm = RAW_PRICES[ASSETS] / RAW_PRICES[ASSETS].iloc[0] * 100
qs   = {k: norm.quantile(v, axis=1) for k, v in
        dict(p10=.10, p25=.25, p50=.50, p75=.75, p90=.90).items()}
fig2 = go.Figure()
for lo, hi, a in [("p10","p90","0.10"), ("p25","p75","0.20")]:
    xv = qs[lo].index.tolist() + qs[hi].index.tolist()[::-1]
    yv = qs[lo].tolist()       + qs[hi].tolist()[::-1]
    fig2.add_trace(go.Scatter(x=xv, y=yv, fill="toself",
        fillcolor=f"rgba(46,117,182,{a})", line=dict(width=0),
        name=f"{lo[1:]}th-{hi[1:]}th pct", hoverinfo="skip"))
fig2.add_trace(go.Scatter(x=qs["p50"].index, y=qs["p50"], mode="lines",
    line=dict(color="#2E75B6", width=2.5), name="Median"))
for t in norm.iloc[-1].nlargest(3).index:
    fig2.add_trace(go.Scatter(x=norm.index, y=norm[t], mode="lines",
        line=dict(width=1, dash="dot"), name=t, opacity=0.7))
for t in norm.iloc[-1].nsmallest(3).index:
    fig2.add_trace(go.Scatter(x=norm.index, y=norm[t], mode="lines",
        line=dict(width=1, dash="dot"), name=t, opacity=0.7))
fig2.update_layout(
    title=f"<b>Normalised Prices (Base=100) — {N} assets</b>",
    xaxis_title="Date", yaxis_title="Normalised Price",
    hovermode="x unified", template="plotly_white", height=480)
fig2.show()

# 3. Return and volatility distributions
ar = RETS.mean() * 252 * 100
av = RETS.std()  * np.sqrt(252) * 100
fig3 = make_subplots(1, 2,
    subplot_titles=["Annualised Return Distribution",
                    "Annualised Volatility Distribution"])
fig3.add_trace(go.Histogram(x=ar, nbinsx=55,
    marker_color="#2A9D8F", opacity=0.8), row=1, col=1)
fig3.add_trace(go.Histogram(x=av, nbinsx=55,
    marker_color="#E63946", opacity=0.8), row=1, col=2)
fig3.add_vline(x=ar.mean(), line=dict(color="#1F3864", dash="dash", width=2),
    annotation_text=f"Mean {ar.mean():.1f}%", row=1, col=1)
fig3.add_vline(x=av.mean(), line=dict(color="#1F3864", dash="dash", width=2),
    annotation_text=f"Mean {av.mean():.1f}%", row=1, col=2)
fig3.update_layout(title=f"<b>Cross-Sectional Statistics — {N} assets</b>",
    template="plotly_white", height=370, showlegend=False)
fig3.show()


---
## Phase 2 · Markowitz Baseline

**Mean-variance problem**

$$\min_w\; w^\top\Sigma w \quad\text{s.t.}\quad \mathbf{1}^\top w=1,\; 0\le w\le w_{\max}$$

- **GMV** — Global Minimum Variance: leftmost point of the frontier.
- **Max-Sharpe** — tangency portfolio on the Capital Market Line.

> With T/N < 5 the sample covariance is singular. We use Ledoit-Wolf throughout.


In [34]:
# ── Core maths ───────────────────────────────────────────────

def annualise(rets, f=252):
    return rets.mean().values * f, rets.cov().values * f

def psd_fix(M, eps=1e-8):
    v, Q = np.linalg.eigh(M)
    return Q @ np.diag(np.maximum(v, eps)) @ Q.T

def pstats(w, mu, cov, rf=0.0):
    w = np.asarray(w, float)
    r = float(w @ mu)
    v = float(max(w @ cov @ w, 1e-12)) ** 0.5
    return dict(ret=r, vol=v, sharpe=(r - rf) / v)

def solve(prob, w_var, fallback=None):
    for solver in [cp.CLARABEL, cp.SCS]:
        try:
            prob.solve(solver=solver, warm_start=True)
            if w_var.value is not None:
                wv = np.clip(w_var.value, 0, None)
                s  = wv.sum()
                return wv / s if s > 1e-10 else fallback
        except cp.SolverError:
            continue
    return fallback

# ── Covariance estimators ─────────────────────────────────────

def cov_sample(rets, f=252):
    return psd_fix(rets.cov().values * f)

def cov_lw(rets, f=252):
    return psd_fix(LedoitWolf().fit(rets.values).covariance_ * f)

def cov_ewm(rets, halflife=60, f=252):
    # Fast vectorised EWM: avoids pandas .ewm().cov() loop (too slow for N>200)
    X = rets.values.astype(float)
    T_ = X.shape[0]
    lam = np.log(2.0) / halflife
    w = np.exp(-lam * np.arange(T_-1, -1, -1, dtype=float))
    w /= w.sum()
    mu_ew = w @ X
    Xc    = X - mu_ew
    C     = (Xc * w[:, None]).T @ Xc
    return psd_fix(C * f)

def get_cov(rets, method="lw"):
    return {"sample": cov_sample, "ewm": cov_ewm, "lw": cov_lw}[method](rets)

# ── Optimisers ────────────────────────────────────────────────

def opt_gmv(mu, cov, lo=0.0, hi=1.0):
    n = len(mu); w = cp.Variable(n)
    return solve(
        cp.Problem(cp.Minimize(cp.quad_form(w, cp.psd_wrap(cov))),
                   [cp.sum(w)==1, w>=lo, w<=hi]),
        w, np.ones(n)/n)

def opt_maxsharpe(mu, cov, rf=0.0, lo=0.0, hi=1.0):
    n  = len(mu); ex = mu - rf
    if np.all(ex <= 0): return opt_gmv(mu, cov, lo, hi)
    y  = cp.Variable(n)
    wv = solve(
        cp.Problem(cp.Minimize(cp.quad_form(y, cp.psd_wrap(cov))),
                   [ex @ y == 1, y >= 0]),
        y, None)
    if wv is None: return opt_gmv(mu, cov, lo, hi)
    wv = np.maximum(wv, 0); return wv / wv.sum()

def opt_target(mu, cov, tgt, lo=0.0, hi=1.0):
    n = len(mu); w = cp.Variable(n)
    return solve(
        cp.Problem(cp.Minimize(cp.quad_form(w, cp.psd_wrap(cov))),
                   [cp.sum(w)==1, mu@w>=tgt, w>=lo, w<=hi]),
        w, None)

def frontier(mu, cov, n_pts=15, rf=0.0, lo=0.0, hi=1.0):
    w0   = opt_gmv(mu, cov, lo, hi)
    rmin = pstats(w0, mu, cov)["ret"]
    rmax = float(np.percentile(mu, 92)) * 0.98
    rows = []
    for tgt in np.linspace(rmin, rmax, n_pts):
        w = opt_target(mu, cov, tgt, lo, hi)
        if w is not None:
            rows.append(pstats(w, mu, cov, rf))
    return pd.DataFrame(rows)

print("Core functions defined.")


Core functions defined.


In [35]:
# ── Full-universe optimisation ───────────────────────────────
print("Estimating Ledoit-Wolf covariance ...")
MU, _  = annualise(RETS)
COV    = cov_lw(RETS)
LO, HI = 0.0, CFG["weight_cap"]

print("Optimising GMV ...")
W_GMV = opt_gmv(MU, COV, LO, HI)
print("Optimising Max-Sharpe ...")
W_MS  = opt_maxsharpe(MU, COV, rf=CFG["rf"], lo=LO, hi=HI)
W_EW  = np.ones(N) / N

S_GMV = pstats(W_GMV, MU, COV, CFG["rf"])
S_MS  = pstats(W_MS,  MU, COV, CFG["rf"])
S_EW  = pstats(W_EW,  MU, COV, CFG["rf"])

tbl = pd.DataFrame({"GMV": S_GMV, "Max-Sharpe": S_MS, "Equal-Weight": S_EW}).T
tbl[["ret","vol"]] *= 100; tbl.columns = ["Return (%)","Vol (%)","Sharpe"]
print("\n", tbl.round(3).to_string())

print(f"\nTracing frontier ({CFG['frontier_pts']} points) ...")
EF = frontier(MU, COV, n_pts=CFG["frontier_pts"],
              rf=CFG["rf"], lo=LO, hi=HI)
print(f"  {len(EF)} feasible points.")


Estimating Ledoit-Wolf covariance ...
Optimising GMV ...
Optimising Max-Sharpe ...

               Return (%)  Vol (%)  Sharpe
GMV                9.197    9.858   0.527
Max-Sharpe        32.368   14.933   1.900
Equal-Weight       9.092   15.836   0.322

Tracing frontier (15 points) ...
  15 feasible points.


In [36]:
# ── Efficient frontier chart ─────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=EF["vol"]*100, y=EF["ret"]*100, mode="lines+markers",
    marker=dict(color=EF["sharpe"], colorscale="Viridis", size=7,
                colorbar=dict(title="Sharpe", x=1.02), showscale=True),
    line=dict(color="rgba(0,0,0,.1)", width=1), name="Frontier",
    hovertemplate="Vol %{x:.2f}%  Ret %{y:.2f}%  Sharpe %{marker.color:.3f}<extra></extra>"))
for lbl, s, sym, clr in [
    ("GMV",         S_GMV, "star",    "#E63946"),
    ("Max-Sharpe",  S_MS,  "diamond", "#2A9D8F"),
    ("Equal-Weight",S_EW,  "circle",  "#F4A261"),
]:
    fig.add_trace(go.Scatter(
        x=[s["vol"]*100], y=[s["ret"]*100], mode="markers",
        marker=dict(size=15, symbol=sym, color=clr,
                    line=dict(width=2, color="white")),
        name=lbl,
        hovertemplate=f"<b>{lbl}</b>  Sharpe={s['sharpe']:.3f}<extra></extra>"))
v_cml = np.linspace(0, S_MS["vol"]*1.5*100, 60)
fig.add_trace(go.Scatter(x=v_cml, y=CFG["rf"]*100 + S_MS["sharpe"]*v_cml,
    mode="lines", line=dict(dash="dash", color="#2A9D8F", width=1.5), name="CML"))
fig.update_layout(
    title=f"<b>Efficient Frontier — {N} Assets (Ledoit-Wolf)</b>",
    xaxis_title="Annualised Vol (%)", yaxis_title="Annualised Return (%)",
    hovermode="closest", template="plotly_white", height=520,
    legend=dict(x=0.02, y=0.98))
fig.show()


In [37]:
# ── Top-10 weights ────────────────────────────────────────────
for lbl, wts in [("GMV", W_GMV), ("Max-Sharpe", W_MS)]:
    top = pd.Series(wts, index=ASSETS).nlargest(10)
    fig_w = go.Figure(go.Bar(
        x=top.values*100, y=top.index, orientation="h",
        marker_color="#2E75B6",
        text=[f"{v*100:.2f}%" for v in top.values], textposition="outside"))
    fig_w.update_layout(title=f"<b>{lbl} — Top 10 Weights</b>",
        xaxis_title="Weight (%)", template="plotly_white",
        height=360, yaxis=dict(autorange="reversed"))
    fig_w.show()


---
## Phase 3 · Robust Covariance Estimation

| Estimator | Formula | Best when |
|-----------|---------|-----------|
| Sample | `(1/T) X'X` | T/N > 10 |
| EWM | recent obs weighted by `exp(-λt)` | regime shifts |
| Ledoit-Wolf | shrink toward `λI` with optimal intensity | T/N < 5 (our case) |

> T/N ≈ 2-3 here — sample covariance is near-singular. Ledoit-Wolf is mandatory.


In [38]:
# ── Compare three estimators ─────────────────────────────────
print("Comparing covariance estimators ...\n")

COV_STORE = {}
rows = []
for lbl, method in [("Sample","sample"), ("EWM hl=60","ewm"), ("Ledoit-Wolf","lw")]:
    t0  = time.time()
    cov = get_cov(RETS, method)
    COV_STORE[method] = cov
    w   = opt_gmv(MU, cov, LO, HI)
    s   = pstats(w, MU, cov, CFG["rf"])
    ev  = np.linalg.eigvalsh(cov)
    cnd = ev.max() / max(ev.min(), 1e-10)
    elapsed = time.time() - t0
    rows.append({
        "Estimator"   : lbl,
        "Return (%)"  : round(s["ret"]*100, 3),
        "Vol (%)"     : round(s["vol"]*100, 3),
        "Sharpe"      : round(s["sharpe"],  4),
        "Min eigenval": round(ev.min(),      6),
        "Cond number" : round(cnd,           1),
        "Time (s)"    : round(elapsed,       2),
    })
    print(f"  {lbl:<18} Sharpe={s['sharpe']:.3f}  cond={cnd:.0f}  ({elapsed:.1f}s)")

print("\n" + pd.DataFrame(rows).set_index("Estimator").to_string())
print("\nVerdict: Ledoit-Wolf — lowest condition number, best out-of-sample stability.")


Comparing covariance estimators ...

  Sample             Sharpe=0.519  cond=103153  (1.2s)
  EWM hl=60          Sharpe=1.230  cond=2386184  (0.7s)
  Ledoit-Wolf        Sharpe=0.527  cond=6769  (1.5s)

             Return (%)  Vol (%)  Sharpe  Min eigenval  Cond number  Time (s)
Estimator                                                                    
Sample            9.141    9.915  0.5186      0.000123     103153.5      1.15
EWM hl=60        11.714    6.274  1.2295      0.000004    2386184.5      0.65
Ledoit-Wolf       9.197    9.858  0.5271      0.001841       6768.6      1.50

Verdict: Ledoit-Wolf — lowest condition number, best out-of-sample stability.


In [39]:
# ── Correlation heatmap (top 30 by GMV weight) ───────────────
top30 = np.argsort(W_GMV)[-30:][::-1]
names30 = [ASSETS[i] for i in top30]

def corr_of(cov):
    d = np.sqrt(np.diag(cov))
    c = cov / np.outer(d, d)
    np.fill_diagonal(c, 1.0)
    return np.round(c, 3)

fig = make_subplots(1, 2,
    subplot_titles=["Sample Correlation", "Ledoit-Wolf Correlation"],
    horizontal_spacing=0.10)
for ci, method in enumerate(["sample","lw"], 1):
    sub  = COV_STORE[method][np.ix_(top30, top30)]
    corr = corr_of(sub)
    fig.add_trace(go.Heatmap(z=corr, x=names30, y=names30,
        colorscale="RdBu_r", zmid=0, zmin=-1, zmax=1,
        showscale=(ci==2), colorbar=dict(title="rho", x=1.02)),
        row=1, col=ci)
fig.update_layout(
    title="<b>Correlation Structure: Sample vs Ledoit-Wolf (Top-30 GMV)</b>",
    template="plotly_white", height=500)
fig.show()


---
## Phase 4 · Constraints & Transaction Costs

**Post-trade net return** = gross return − 2 × TC_bps/10000 × one-way turnover

Linear transaction costs and turnover constraints are convex and fit natively in CVXPY.


In [40]:
# ── Constrained optimiser ─────────────────────────────────────
def opt_constrained(mu, cov, prev_w=None,
                    lo=0.0, hi=0.05, max_tov=0.30,
                    tc_bps=10, rf=0.0):
    n    = len(mu)
    tc   = tc_bps / 10_000
    prev = np.asarray(prev_w) if prev_w is not None else np.ones(n)/n
    w    = cp.Variable(n)
    tov  = cp.sum(cp.abs(w - prev)) / 2
    wv   = solve(
        cp.Problem(cp.Minimize(cp.quad_form(w, cp.psd_wrap(cov))),
                   [cp.sum(w)==1, w>=lo, w<=hi, tov<=max_tov]),
        w, prev)
    tov_v   = float(np.sum(np.abs(wv - prev))) / 2
    tc_drag = tov_v * 2 * tc
    s       = pstats(wv, mu, cov, rf)
    return dict(weights=wv, stats=s,
                turnover=tov_v, tc_drag=tc_drag,
                net_ret=s["ret"] - tc_drag)


res_uncon = dict(weights=W_GMV, stats=S_GMV, turnover=0.0,
                 tc_drag=0.0, net_ret=S_GMV["ret"])
res_con   = opt_constrained(MU, COV, prev_w=W_EW,
                             lo=0.0, hi=CFG["weight_cap"],
                             max_tov=0.30, tc_bps=CFG["tc_bps"],
                             rf=CFG["rf"])

hdr = f"  {'':20}  {'Return':>9}  {'Vol':>9}  {'Sharpe':>8}  {'Turnover':>9}  {'TC drag':>8}"
print(hdr)
for lbl, r in [("Unconstrained GMV", res_uncon), ("Constrained GMV", res_con)]:
    s = r["stats"]
    print(f"  {lbl:20}  {s['ret']*100:>8.2f}%  {s['vol']*100:>8.2f}%  "
          f"{s['sharpe']:>8.3f}  {r['turnover']*100:>8.1f}%  {r['tc_drag']*100:>7.3f}%")

# Weight comparison chart
top20 = np.argsort(W_GMV)[-20:][::-1]
t20n  = [ASSETS[i] for i in top20]
fig   = go.Figure()
for lbl, wts, col in [
    ("Unconstrained GMV", W_GMV,              "#E63946"),
    ("Constrained GMV",   res_con["weights"], "#2A9D8F"),
    ("Equal Weight",      W_EW,               "#F4A261"),
]:
    fig.add_trace(go.Bar(x=t20n, y=[wts[i]*100 for i in top20],
        name=lbl, marker_color=col, opacity=0.85))
fig.add_hline(y=CFG["weight_cap"]*100,
              line=dict(dash="dot", color="grey", width=1.5),
              annotation_text=f"Cap {CFG['weight_cap']*100:.0f}%")
fig.update_layout(
    title="<b>Weight Comparison — Top 20 GMV Assets</b>",
    barmode="group", xaxis_title="Asset", yaxis_title="Weight (%)",
    template="plotly_white", height=420)
fig.show()


                           Return        Vol    Sharpe   Turnover   TC drag
  Unconstrained GMV         9.20%      9.86%     0.527       0.0%    0.000%
  Constrained GMV           9.00%     11.49%     0.435      30.0%    0.060%


---
## Phase 5 · Rolling-Window Backtest

Rules that prevent look-ahead bias:
- Training window uses only data **before** the rebalance date.
- Transaction costs are deducted at every rebalance.
- All strategies use the same dates and cost model.

> Runtime: ~5-15 min on Colab depending on GPU/CPU tier.


In [41]:
# ── Backtest engine ───────────────────────────────────────────
class RollingBacktest:
    def __init__(self, rets, train=252, freq=21,
                 rf_daily=None, tc_bps=10, lo=0.0, hi=0.05):
        self.arr   = rets.values
        self.dates = rets.index
        self.cols  = rets.columns.tolist()
        self.T, self.N = rets.shape
        self.tw    = train
        self.freq  = freq
        self.rf    = rf_daily if rf_daily is not None else CFG["rf"]/252
        self.tc    = tc_bps / 10_000
        self.lo    = lo; self.hi = hi
        self.res   = {}

    def _fn(self, name):
        lo, hi = self.lo, self.hi
        def eq(df, _pw):   return np.ones(self.N)/self.N
        def gmv_s(df, _pw):
            mu, _ = annualise(df); return opt_gmv(mu, cov_sample(df), lo, hi)
        def gmv_lw(df, _pw):
            mu, _ = annualise(df); return opt_gmv(mu, cov_lw(df), lo, hi)
        def gmv_ew(df, _pw):
            mu, _ = annualise(df); return opt_gmv(mu, cov_ewm(df), lo, hi)
        def ms_lw(df, _pw):
            mu, _ = annualise(df)
            return opt_maxsharpe(mu, cov_lw(df), rf=self.rf*252, lo=lo, hi=hi)
        def con_lw(df, pw):
            mu, _ = annualise(df)
            r = opt_constrained(mu, cov_lw(df), prev_w=pw,
                                lo=lo, hi=hi, max_tov=0.30,
                                tc_bps=self.tc*10_000, rf=self.rf*252)
            return r["weights"]
        return {"Equal Weight"    : eq,
                "GMV (Sample)"    : gmv_s,
                "GMV (LW)"        : gmv_lw,
                "GMV (EWM)"       : gmv_ew,
                "Max-Sharpe (LW)" : ms_lw,
                "Constrained (LW)": con_lw}[name]

    def run(self, strategies=None):
        if strategies is None:
            strategies = ["Equal Weight","GMV (Sample)","GMV (LW)",
                          "GMV (EWM)","Max-Sharpe (LW)","Constrained (LW)"]
        for strat in strategies:
            print(f"  {strat:<24}", end="  ")
            fn  = self._fn(strat)
            w   = np.ones(self.N)/self.N
            out = []
            t0  = time.time()
            for t in range(self.tw, self.T):
                if (t - self.tw) % self.freq == 0:
                    df = pd.DataFrame(self.arr[t-self.tw:t], columns=self.cols)
                    try:
                        nw = fn(df, w)
                        if nw is None or not np.isfinite(nw).all(): nw = w
                    except Exception: nw = w
                    tov = float(np.sum(np.abs(nw - w)))/2
                    drag = tov * 2 * self.tc
                    w = nw.copy()
                else: drag = 0.0
                out.append(float(w @ self.arr[t]) - drag)
            idx = self.dates[self.tw:]
            rs  = pd.Series(out, index=idx, name=strat)
            cum = (1+rs).cumprod()
            self.res[strat] = dict(ret=rs, cum=cum)
            print(f"done  ({time.time()-t0:.0f}s)  cum={cum.iloc[-1]-1:.1%}")
        return self

    def metrics(self):
        rows = []
        rf252 = self.rf * 252
        for nm, d in self.res.items():
            r   = d["ret"]; c = d["cum"]
            ar  = (1+r.mean())**252-1
            av  = r.std()*252**0.5
            sp  = (ar-rf252)/av
            dnv = r[r<0].std()*252**0.5
            so  = ar/max(dnv,1e-10)
            dd  = (c/c.cummax()-1).min()
            v95 = np.percentile(r,5)
            cv  = r[r<=v95].mean()
            rows.append({"Strategy":nm,
                "Ann. Return (%)":round(ar*100,2), "Ann. Vol (%)":round(av*100,2),
                "Sharpe":round(sp,3), "Sortino":round(so,3),
                "Max DD (%)":round(dd*100,2),
                "VaR 95% (%)":round(v95*100,3), "CVaR 95% (%)":round(cv*100,3)})
        return pd.DataFrame(rows).set_index("Strategy")

    def _col(self, i): return px.colors.qualitative.Set1[i%9]

    def plot_cum(self):
        fig = go.Figure()
        for i,(nm,d) in enumerate(self.res.items()):
            c = d["cum"]
            fig.add_trace(go.Scatter(x=c.index, y=(c-1)*100, name=nm,
                mode="lines", line=dict(color=self._col(i), width=2)))
        fig.update_layout(title="<b>Out-of-Sample Cumulative Returns</b>",
            xaxis_title="Date", yaxis_title="Cum. Return (%)",
            hovermode="x unified", template="plotly_white", height=480)
        fig.show()

    def plot_dd(self):
        fills = ["rgba(228,26,28,.10)","rgba(55,126,184,.10)",
                 "rgba(77,175,74,.10)","rgba(152,78,163,.10)",
                 "rgba(255,127,0,.10)","rgba(166,86,40,.10)"]
        fig = go.Figure()
        for i,(nm,d) in enumerate(self.res.items()):
            c = d["cum"]; dd = (c/c.cummax()-1)*100
            fig.add_trace(go.Scatter(x=dd.index, y=dd, name=nm,
                mode="lines", fill="tozeroy",
                fillcolor=fills[i%len(fills)],
                line=dict(color=self._col(i), width=1.5)))
        fig.update_layout(title="<b>Portfolio Drawdowns</b>",
            xaxis_title="Date", yaxis_title="Drawdown (%)",
            hovermode="x unified", template="plotly_white", height=400)
        fig.show()

    def plot_vol(self, w=21):
        fig = go.Figure()
        for i,(nm,d) in enumerate(self.res.items()):
            rv = d["ret"].rolling(w).std()*252**0.5*100
            fig.add_trace(go.Scatter(x=rv.index, y=rv, name=nm,
                mode="lines", line=dict(color=self._col(i), width=1.5)))
        fig.update_layout(title=f"<b>Rolling {w}-Day Annualised Vol</b>",
            xaxis_title="Date", yaxis_title="Vol (%)",
            hovermode="x unified", template="plotly_white", height=400)
        fig.show()

    def plot_heat(self, strategy=None):
        nm  = strategy or list(self.res.keys())[2]
        mo  = (self.res[nm]["ret"]+1).resample("ME").prod()-1
        df  = mo.to_frame("r")
        df["y"] = df.index.year; df["m"] = df.index.month
        piv = df.pivot(index="y", columns="m", values="r")*100
        piv.columns = ["Jan","Feb","Mar","Apr","May","Jun",
                       "Jul","Aug","Sep","Oct","Nov","Dec"][:len(piv.columns)]
        fig = go.Figure(go.Heatmap(
            z=piv.values, x=piv.columns.tolist(), y=piv.index.tolist(),
            colorscale="RdYlGn", zmid=0,
            text=np.round(piv.values,1).astype(str), texttemplate="%{text}%",
            colorbar=dict(title="Return %")))
        fig.update_layout(title=f"<b>Monthly Returns — {nm}</b>",
            template="plotly_white", height=max(280, 60*len(piv)))
        fig.show()

print("RollingBacktest defined.")


RollingBacktest defined.


In [42]:
# ── Run backtest ──────────────────────────────────────────────
BT = RollingBacktest(RETS,
                     train  = CFG["train_window"],
                     freq   = CFG["rebal_freq"],
                     tc_bps = CFG["tc_bps"],
                     lo=0.0, hi=CFG["weight_cap"])
BT.run()

print("\n" + "="*60)
METRICS = BT.metrics()
print(METRICS.to_string())


  Equal Weight              done  (0s)  cum=16.6%
  GMV (Sample)              done  (22s)  cum=24.7%
  GMV (LW)                  done  (29s)  cum=23.9%
  GMV (EWM)                 done  (18s)  cum=21.7%
  Max-Sharpe (LW)           done  (30s)  cum=62.4%
  Constrained (LW)          done  (37s)  cum=21.9%

                  Ann. Return (%)  Ann. Vol (%)  Sharpe  Sortino  Max DD (%)  VaR 95% (%)  CVaR 95% (%)
Strategy                                                                                               
Equal Weight                 5.11         16.49   0.067    0.437      -22.17       -1.576        -2.379
GMV (Sample)                 5.96         10.81   0.182    0.750      -17.13       -1.092        -1.562
GMV (LW)                     5.80         10.82   0.167    0.726      -17.46       -1.107        -1.567
GMV (EWM)                    5.37         10.97   0.125    0.656      -18.72       -1.088        -1.598
Max-Sharpe (LW)             13.74         17.05   0.571    1.104      

In [43]:
BT.plot_cum()
BT.plot_dd()
BT.plot_vol()
BT.plot_heat()


---
## Phase 6 · CVaR-Constrained Portfolio

CVaR (Expected Shortfall) = average loss **beyond** the VaR threshold. It is a
coherent risk measure; variance is not.

**Linearisation — Rockafellar & Uryasev (2000)**

$$\min_w\; w^\top\Sigma w \quad\text{s.t.}\quad
\gamma + \frac{1}{\alpha T}\sum_t z_t \le c,\quad
z_t \ge -r_t^\top w - \gamma,\quad z_t \ge 0$$


In [44]:
# ── CVaR helpers ─────────────────────────────────────────────
def hist_cvar(rets_df, w, alpha=0.05):
    pr  = rets_df.values @ w
    thr = np.percentile(pr, alpha*100)
    return -thr, -pr[pr <= thr].mean()

def opt_cvar(rets_df, mu, cov, cvar_limit=None,
             alpha=0.05, lo=0.0, hi=0.05, rf=0.0):
    sc = rets_df.values; Ts, n = sc.shape
    w  = cp.Variable(n); gamma = cp.Variable(); z = cp.Variable(Ts)
    cvar_expr = gamma + (1.0/(alpha*Ts))*cp.sum(z)
    cons = [cp.sum(w)==1, w>=lo, w<=hi,
            z>=0, z >= -(sc @ w) - gamma]
    if cvar_limit is not None:
        cons.append(cvar_expr <= cvar_limit)
    wv = None
    for slv in [cp.CLARABEL, cp.SCS]:
        try:
            cp.Problem(cp.Minimize(cp.quad_form(w, cp.psd_wrap(cov))), cons).solve(
                solver=slv, warm_start=True)
            if w.value is not None:
                wv = np.clip(w.value, 0, None); wv /= wv.sum(); break
        except cp.SolverError: continue
    if wv is None: return None
    var_, cvar_ = hist_cvar(rets_df, wv, alpha)
    return dict(weights=wv, stats=pstats(wv, mu, cov, rf),
                var=var_, cvar=cvar_, eff_n=1.0/np.sum(wv**2))

print("CVaR functions defined.")


CVaR functions defined.


In [45]:
# ── CVaR sensitivity sweep ────────────────────────────────────
cov_lw_full = COV_STORE.get("lw", cov_lw(RETS))
_, base_cvar = hist_cvar(RETS, W_GMV)
print(f"Baseline GMV daily CVaR 95% : {base_cvar*100:.4f}%\n")

configs = {
    "No CVaR constraint"      : None,
    "CVaR <= 1.20x baseline"  : base_cvar * 1.20,
    "CVaR <= 1.00x baseline"  : base_cvar * 1.00,
    "CVaR <= 0.90x baseline"  : base_cvar * 0.90,
}
cvar_rows = []; CVAR_W = {}
for lbl, lim in configs.items():
    print(f"  {lbl} ...", end="  ")
    res = opt_cvar(RETS, MU, cov_lw_full,
                   cvar_limit=lim, alpha=0.05,
                   lo=0.0, hi=CFG["weight_cap"], rf=CFG["rf"])
    if res is None: print("failed"); continue
    s = res["stats"]; CVAR_W[lbl] = res["weights"]
    cvar_rows.append({"Config":lbl,
        "Return (%)":round(s["ret"]*100,3), "Vol (%)":round(s["vol"]*100,3),
        "Sharpe":round(s["sharpe"],3),
        "CVaR daily (%)":round(res["cvar"]*100,4),
        "Eff. N":round(res["eff_n"],1)})
    print(f"Sharpe={s['sharpe']:.3f}  CVaR={res['cvar']*100:.4f}%  EffN={res['eff_n']:.1f}")

print("\n" + pd.DataFrame(cvar_rows).set_index("Config").to_string())
print("\nNote: tighter CVaR reduces Eff.N -> portfolio may concentrate.")


Baseline GMV daily CVaR 95% : 1.3996%

  No CVaR constraint ...  Sharpe=0.527  CVaR=1.3996%  EffN=32.5
  CVaR <= 1.20x baseline ...  Sharpe=0.527  CVaR=1.3996%  EffN=32.5
  CVaR <= 1.00x baseline ...  Sharpe=0.523  CVaR=1.3938%  EffN=32.5
  CVaR <= 0.90x baseline ...  failed

                        Return (%)  Vol (%)  Sharpe  CVaR daily (%)  Eff. N
Config                                                                     
No CVaR constraint           9.197    9.858   0.527          1.3996    32.5
CVaR <= 1.20x baseline       9.197    9.858   0.527          1.3996    32.5
CVaR <= 1.00x baseline       9.158    9.860   0.523          1.3938    32.5

Note: tighter CVaR reduces Eff.N -> portfolio may concentrate.


In [46]:
# ── CVaR weight chart ────────────────────────────────────────
fig = go.Figure()
cols = px.colors.qualitative.Pastel1
for i,(lbl,wts) in enumerate(CVAR_W.items()):
    top = np.argsort(wts)[-20:][::-1]
    fig.add_trace(go.Bar(x=[ASSETS[j] for j in top],
        y=[wts[j]*100 for j in top], name=lbl,
        marker_color=cols[i%len(cols)], opacity=0.88))
fig.add_hline(y=CFG["weight_cap"]*100,
              line=dict(dash="dot", color="grey"),
              annotation_text="Weight cap")
fig.update_layout(
    title="<b>CVaR Constraint Comparison — Top-20 Weights</b>",
    barmode="group", xaxis_title="Asset", yaxis_title="Weight (%)",
    template="plotly_white", height=450)
fig.show()


---
## Phase 7 · Stress Testing

1. **Historical** — worst rolling windows in the actual return series.
2. **Hypothetical** — shock returns, volatilities, and correlations to extreme values.


In [47]:
# ── Stress functions ─────────────────────────────────────────
def worst_windows(port_r, windows=(5, 21, 63, 126)):
    rows = []
    for w in windows:
        roll = port_r.rolling(w).sum()
        loc  = roll.argmin()
        rows.append({
            "Window"       : f"{w}d",
            "Worst Ret (%)": round(roll.min()*100, 2),
            "Start"        : str(roll.index[max(0,loc-w+1)].date()),
            "End"          : str(roll.index[loc].date()),
        })
    return pd.DataFrame(rows)

def scenario(w, mu, cov,
             ret_shock=None, vol_mult=1.0, corr_level=None):
    mu2 = mu.copy(); cov2 = cov.copy()
    if ret_shock is not None: mu2[:] = ret_shock
    if vol_mult != 1.0:
        std  = np.sqrt(np.diag(cov2)); std2 = std*vol_mult
        corr = cov2/np.outer(std,std); cov2 = corr*np.outer(std2,std2)
    if corr_level is not None:
        std  = np.sqrt(np.diag(cov2))
        corr = cov2/np.outer(std,std)
        mask = ~np.eye(len(corr), dtype=bool)
        corr[mask] = np.clip(corr_level, -0.99, 0.99)
        np.fill_diagonal(corr, 1.0); cov2 = corr*np.outer(std,std)
    cov2 = psd_fix(cov2)
    s0   = pstats(w, mu,  cov)
    s1   = pstats(w, mu2, cov2)
    return {"Base Ret (%)": round(s0["ret"]*100, 2),
            "Shocked (%)": round(s1["ret"]*100, 2),
            "dRet (%)": round((s1["ret"]-s0["ret"])*100, 2),
            "dVol (%)": round((s1["vol"]-s0["vol"])*100, 2),
            "dSharpe" : round(s1["sharpe"]-s0["sharpe"], 3)}

SCENARIOS = {
    "Equity crash -20%"      : dict(ret_shock=-0.20),
    "Vol spike x3"           : dict(vol_mult=3.0),
    "Corr spike to 0.80"     : dict(corr_level=0.80),
    "Crash + Vol spike"      : dict(ret_shock=-0.15, vol_mult=2.0),
    "Deflation ret=0"        : dict(ret_shock=0.00),
    "Stagflation -10% vol x2": dict(ret_shock=-0.10, vol_mult=2.0),
    "Corr collapse to 0"     : dict(corr_level=0.00),
}
print("Stress functions defined.")


Stress functions defined.


In [48]:
# ── Run stress tests ─────────────────────────────────────────
BEST_STRAT = METRICS["Sharpe"].idxmax()
BEST_RETS  = BT.res[BEST_STRAT]["ret"]

print("HISTORICAL WORST WINDOWS")
print(worst_windows(BEST_RETS).to_string(index=False))

print("\nSCENARIO ANALYSIS (GMV weights)")
STRESS = {nm: scenario(W_GMV, MU, COV, **kw) for nm, kw in SCENARIOS.items()}
print(pd.DataFrame(STRESS).T.to_string())


HISTORICAL WORST WINDOWS
Window  Worst Ret (%)      Start        End
    5d         -11.91 2025-04-02 2025-04-08
   21d         -13.16 2022-04-08 2022-05-09
   63d         -18.02 2022-04-11 2022-07-12
  126d         -14.83 2022-04-11 2022-10-10

SCENARIO ANALYSIS (GMV weights)
                         Base Ret (%)  Shocked (%)  dRet (%)  dVol (%)  dSharpe
Equity crash -20%                 9.2        -20.0     -29.2      0.00   -2.962
Vol spike x3                      9.2          9.2       0.0     19.72   -0.622
Corr spike to 0.80                9.2          9.2       0.0     11.45   -0.501
Crash + Vol spike                 9.2        -15.0     -24.2      9.86   -1.694
Deflation ret=0                   9.2          0.0      -9.2      0.00   -0.933
Stagflation -10% vol x2           9.2        -10.0     -19.2      9.86   -1.440
Corr collapse to 0                9.2          9.2       0.0     -5.87    1.373


In [49]:
# ── Stress visualisations ────────────────────────────────────

# 1. Scenario bar chart
d_ret = [v["dRet (%)"] for v in STRESS.values()]
d_vol = [v["dVol (%)"] for v in STRESS.values()]
snames = list(STRESS.keys())

fig = make_subplots(1, 2,
    subplot_titles=["Return Impact (%)", "Vol Impact (%)"],
    horizontal_spacing=0.14)
fig.add_trace(go.Bar(x=snames, y=d_ret,
    marker_color=["#E63946" if v<0 else "#2A9D8F" for v in d_ret],
    name="dRet"), row=1, col=1)
fig.add_trace(go.Bar(x=snames, y=d_vol,
    marker_color=["#E63946" if v>0 else "#2A9D8F" for v in d_vol],
    name="dVol"), row=1, col=2)
fig.update_layout(title="<b>Stress Impact — GMV Portfolio</b>",
    template="plotly_white", height=440, showlegend=False)
fig.update_xaxes(tickangle=30)
fig.show()

# 2. Return distribution with VaR/CVaR
v95  = np.percentile(BEST_RETS, 5)
cv95 = BEST_RETS[BEST_RETS <= v95].mean()
fig2 = go.Figure()
fig2.add_trace(go.Histogram(x=BEST_RETS*100, nbinsx=80,
    marker_color="#2A9D8F", opacity=0.75, name="Daily Returns"))
fig2.add_vline(x=v95*100,  line=dict(color="#F4A261", dash="dash", width=2),
    annotation_text=f"VaR 95% {v95*100:.3f}%")
fig2.add_vline(x=cv95*100, line=dict(color="#E63946", dash="dot",  width=2),
    annotation_text=f"CVaR 95% {cv95*100:.3f}%",
    annotation_position="top left")
fig2.update_layout(
    title=f"<b>Return Distribution — {BEST_STRAT}</b>",
    xaxis_title="Daily Return (%)", yaxis_title="Count",
    template="plotly_white", height=390)
fig2.show()

# 3. Underwater curve
cum = BT.res[BEST_STRAT]["cum"]
dd  = (cum/cum.cummax()-1)*100
fig3 = go.Figure(go.Scatter(x=dd.index, y=dd, fill="tozeroy",
    fillcolor="rgba(230,57,70,0.12)",
    line=dict(color="#E63946", width=1.5), name="Drawdown"))
fig3.update_layout(
    title=f"<b>Drawdown — {BEST_STRAT}</b>",
    xaxis_title="Date", yaxis_title="Drawdown (%)",
    template="plotly_white", height=360)
fig3.show()


---
## Phase 8 · Final Result

Best strategy by out-of-sample Sharpe ratio — final weights, full metrics, dashboard.


In [50]:
# ── Recover final weights ────────────────────────────────────
METRICS  = BT.metrics()
BEST     = METRICS["Sharpe"].idxmax()
BEST_ROW = METRICS.loc[BEST]

fn_  = BT._fn(BEST)
w_   = np.ones(N)/N
for t_ in range(BT.tw, BT.T):
    if (t_ - BT.tw) % BT.freq == 0:
        df_ = pd.DataFrame(BT.arr[t_-BT.tw:t_], columns=BT.cols)
        try:
            nw_ = fn_(df_, w_)
            if nw_ is not None and np.isfinite(nw_).all(): w_ = nw_
        except Exception: pass
FINAL_W  = w_
FINAL_WT = pd.Series(FINAL_W, index=ASSETS).sort_values(ascending=False)

# Console banner
br = BEST_ROW
lines = [
    "=" * 58,
    "  RECOMMENDED PORTFOLIO",
   f"  Strategy  : {BEST}",
    "-" * 58,
   f"  Ann. Return   : {br['Ann. Return (%)']:>7.2f} %",
   f"  Ann. Vol      : {br['Ann. Vol (%)']:>7.2f} %",
   f"  Sharpe        : {br['Sharpe']:>7.3f}",
   f"  Sortino       : {br['Sortino']:>7.3f}",
   f"  Max Drawdown  : {br['Max DD (%)']:>7.2f} %",
   f"  VaR  95%      : {br['VaR 95% (%)']:>7.3f} %  (daily)",
   f"  CVaR 95%      : {br['CVaR 95% (%)']:>7.3f} %  (daily)",
    "=" * 58,
]
print("\n".join(lines))

print("\n  TOP 20 WEIGHTS\n")
for tick, wt in FINAL_WT.head(20).items():
    bar = "#" * int(wt * 300)
    print(f"  {tick:<8}  {wt*100:>5.2f}%  {bar}")

eff_n = 1.0 / np.sum(FINAL_W**2)
nhold = int((FINAL_W > 0.005).sum())
print(f"\n  Assets held : {nhold} / {N}   Eff.N : {eff_n:.1f}")
print(f"  Largest     : {FINAL_WT.index[0]}  {FINAL_WT.iloc[0]*100:.2f}%")
print(f"  Weight sum  : {FINAL_W.sum():.6f}")
print("\n  ALL STRATEGIES RANKED BY SHARPE")
print(METRICS.sort_values("Sharpe", ascending=False).to_string())


  RECOMMENDED PORTFOLIO
  Strategy  : Max-Sharpe (LW)
----------------------------------------------------------
  Ann. Return   :   13.74 %
  Ann. Vol      :   17.05 %
  Sharpe        :   0.571
  Sortino       :   1.104
  Max Drawdown  :  -19.49 %
  VaR  95%      :  -1.806 %  (daily)
  CVaR 95%      :  -2.517 %  (daily)

  TOP 20 WEIGHTS

  JNJ        9.93%  #############################
  CF         9.24%  ###########################
  GOOG       8.51%  #########################
  DG         7.61%  ######################
  OHI        7.52%  ######################
  RCI        6.96%  ####################
  HCA        6.23%  ##################
  STX        5.06%  ###############
  CAH        4.93%  ##############
  CME        4.37%  #############
  NEM        4.23%  ############
  MO         3.61%  ##########
  NOC        3.58%  ##########
  PWR        3.35%  ##########
  MU         2.92%  ########
  ROST       2.51%  #######
  GLW        2.22%  ######
  LHX        1.45%  ####
  WBD   

In [51]:
# ── Dashboard ────────────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Final Portfolio — Top 20 Weights",
        "Strategy Comparison (Sharpe)",
        "Cumulative Return: Winner vs Equal Weight",
        "Drawdown: Winner vs Equal Weight",
    ],
    vertical_spacing=0.18, horizontal_spacing=0.12)

# 1. Weight bars
top20 = FINAL_WT.head(20)
blues = [f"rgba(46,117,182,{0.45+0.55*v/top20.max():.2f})" for v in top20]
fig.add_trace(go.Bar(
    x=top20.values*100, y=top20.index, orientation="h",
    marker_color=blues,
    text=[f"{v*100:.2f}%" for v in top20], textposition="outside",
    name="Weight"), row=1, col=1)

# 2. Sharpe ranking
ranked = METRICS.sort_values("Sharpe")
bclr   = ["#E63946" if nm==BEST else "#2E75B6" for nm in ranked.index]
fig.add_trace(go.Bar(
    y=ranked.index, x=ranked["Sharpe"], orientation="h",
    marker_color=bclr,
    text=[f"{v:.3f}" for v in ranked["Sharpe"]], textposition="outside",
    name="Sharpe"), row=1, col=2)

# 3. Cumulative returns
for strat, col, dash in [(BEST,"#2E75B6","solid"),("Equal Weight","#F4A261","dash")]:
    c = BT.res[strat]["cum"]
    fig.add_trace(go.Scatter(x=c.index, y=(c-1)*100, name=strat,
        mode="lines", line=dict(color=col, dash=dash, width=2)), row=2, col=1)

# 4. Drawdowns
for strat, col, fill, dash in [
    (BEST,          "#E63946","rgba(230,57,70,0.12)", "solid"),
    ("Equal Weight","#F4A261","rgba(244,162,97,0.12)","dash"),
]:
    c  = BT.res[strat]["cum"]; dd = (c/c.cummax()-1)*100
    fig.add_trace(go.Scatter(x=dd.index, y=dd, name=strat+" DD",
        mode="lines", fill="tozeroy", fillcolor=fill,
        line=dict(color=col, dash=dash, width=1.5), showlegend=False),
        row=2, col=2)

br = BEST_ROW
fig.update_layout(
    title=dict(
        text=(f"<b>Final Portfolio Dashboard — {BEST}</b><br>"
              f"<sup>Sharpe {br['Sharpe']:.3f}  "
              f"Return {br['Ann. Return (%)']:.2f}%  "
              f"Vol {br['Ann. Vol (%)']:.2f}%  "
              f"MaxDD {br['Max DD (%)']:.2f}%</sup>"),
        x=0.5, xanchor="center"),
    template="plotly_white", height=760,
    legend=dict(orientation="h", y=-0.06))
fig.update_xaxes(title_text="Weight (%)",           row=1, col=1)
fig.update_xaxes(title_text="Out-of-Sample Sharpe", row=1, col=2)
fig.update_yaxes(title_text="Cum. Return (%)",      row=2, col=1)
fig.update_yaxes(title_text="Drawdown (%)",         row=2, col=2)
fig.show()


In [52]:
# ── Sector allocation ────────────────────────────────────────
sec_w = {}
for tick, wt in zip(ASSETS, FINAL_W):
    s = SECTOR_MAP.get(tick, "Unknown").replace(" ETF","")
    sec_w[s] = sec_w.get(s, 0) + wt
sw = pd.Series(sec_w).sort_values(ascending=False)
fig = go.Figure(go.Bar(
    x=sw.index, y=sw.values*100,
    marker_color=px.colors.qualitative.Set3[:len(sw)],
    text=[f"{v*100:.1f}%" for v in sw.values], textposition="outside"))
fig.update_layout(
    title=f"<b>Sector Allocation — {BEST}</b>",
    xaxis_title="Sector", yaxis_title="Weight (%)",
    xaxis_tickangle=30, template="plotly_white", height=450)
fig.show()


In [53]:
# ── Radar chart ──────────────────────────────────────────────
m = METRICS.copy()
for col in ["Ann. Vol (%)","Max DD (%)","VaR 95% (%)","CVaR 95% (%)"]:
    m[col] = -m[col]
nr = (m - m.min()) / (m.max() - m.min() + 1e-10)
cats = ["Ann. Return (%)","Ann. Vol (%)","Sharpe","Sortino","Max DD (%)","VaR 95% (%)"]
lbls = ["Return","Low Vol","Sharpe","Sortino","Low DD","Low VaR"]
fig = go.Figure()
for i, strat in enumerate(nr.index):
    vals = nr.loc[strat, cats].tolist(); vals += [vals[0]]
    fig.add_trace(go.Scatterpolar(
        r=vals, theta=lbls+[lbls[0]], fill="toself",
        name=strat, line=dict(color=px.colors.qualitative.Set1[i%9]),
        opacity=0.70))
fig.update_layout(
    title="<b>Strategy Radar — Normalised [0=worst, 1=best]</b>",
    polar=dict(radialaxis=dict(visible=True, range=[0,1])),
    template="plotly_white", height=520, legend=dict(x=1.05, y=0.5))
fig.show()


In [54]:
# ── Strategy correlation heatmap ─────────────────────────────
sr = pd.DataFrame({k: v["ret"] for k,v in BT.res.items()}).dropna()
sc = sr.corr().round(3)
fig = go.Figure(go.Heatmap(
    z=sc.values, x=sc.columns.tolist(), y=sc.index.tolist(),
    colorscale="RdBu_r", zmid=0, zmin=-1, zmax=1,
    text=sc.values, texttemplate="%{text}",
    colorbar=dict(title="rho")))
fig.update_layout(
    title="<b>Strategy Return Correlations (Out-of-Sample)</b>",
    template="plotly_white", height=400)
fig.show()

print("\nPipeline complete.")
print(f"  Winner : {BEST}")
print(f"  Sharpe : {BEST_ROW['Sharpe']:.3f}")
print(f"  Return : {BEST_ROW['Ann. Return (%)']:.2f}%")
print(f"  MaxDD  : {BEST_ROW['Max DD (%)']:.2f}%")



Pipeline complete.
  Winner : Max-Sharpe (LW)
  Sharpe : 0.571
  Return : 13.74%
  MaxDD  : -19.49%
